# Algoritmo CNN

## Fundamentos Matemáticos

* Convolución: un filtro recorre la imagen extrayendo características.
* ReLU: introduce no linealidad.
* Softmax: convierte la salida en probabilidades.

## Métricas

* Accuracy: porcentaje de aciertos.
* Precision: qué tan correctos son los positivos predichos.
* Recall: qué tantos positivos reales detecta.
* F1-Score: balance entre precisión y recall.
* Cross-Entropy Loss: mide el error entre probabilidades predichas y reales.

## Descripción formal o narrativa

Una CNN analiza imágenes buscando patrones como bordes, formas y texturas. Primero extrae características mediante filtros, luego reduce información con pooling y finalmente clasifica usando capas densas.

## Pseudocódigo



In [ ]:
CNN(Imagen)

Aplicar Convolución

Aplicar ReLU

Aplicar Pooling

Aplanar características

Aplicar Capa Densa + Softmax

Retornar probabilidades

## Algoritmo


In [ ]:
import numpy as np

# ==========================================
# 1. Capa de Convolución (Didáctica)
# ==========================================
class CapaConvolucion:
    def __init__(self, num_filtros, tamaño_kernel):
        self.num_filtros = num_filtros
        self.tamaño_kernel = tamaño_kernel
        # Inicialización de filtros al azar, normalizados
        self.filtros = np.random.randn(num_filtros, tamaño_kernel, tamaño_kernel) / (tamaño_kernel**2)

    def _obtener_regiones(self, imagen):
        h, w = imagen.shape
        for i in range(h - self.tamaño_kernel + 1):
            for j in range(w - self.tamaño_kernel + 1):
                region = imagen[i:(i + self.tamaño_kernel), j:(j + self.tamaño_kernel)]
                yield region, i, j

    def forward(self, entrada):
        self.ultima_entrada = entrada
        h, w = entrada.shape
        salida = np.zeros((h - self.tamaño_kernel + 1, w - self.tamaño_kernel + 1, self.num_filtros))

        for region, i, j in self._obtener_regiones(entrada):
            # Operación de convolución: suma del producto punto
            salida[i, j] = np.sum(region * self.filtros, axis=(1, 2))
        return salida

    def backward(self, d_L_d_salida, learning_rate):
        # --- EL RETO MATEMÁTICO ---
        # Calculamos el gradiente del error con respecto a los filtros
        d_L_d_filtros = np.zeros(self.filtros.shape)
        for region, i, j in self._obtener_regiones(self.ultima_entrada):
            for f in range(self.num_filtros):
                d_L_d_filtros[f] += region * d_L_d_salida[i, j, f]

        # Actualizamos los filtros usando descenso de gradiente
        self.filtros -= learning_rate * d_L_d_filtros
        # (Aquí faltaría calcular d_L_d_entrada para propagar el error atrás,
        # pero para una CNN simple de una capa lo omitimos para claridad)
        return None

# ==========================================
# 2. Capa de Max Pooling
# ==========================================
class CapaMaxPooling:
    def __init__(self, tamaño_pool=2):
        self.tamaño_pool = tamaño_pool

    def _obtener_regiones(self, imagen):
        h, w, num_filtros = imagen.shape
        nuevo_h = h // self.tamaño_pool
        nuevo_w = w // self.tamaño_pool
        for i in range(nuevo_h):
            for j in range(nuevo_w):
                region = imagen[(i * self.tamaño_pool):(i * self.tamaño_pool + self.tamaño_pool),
                                (j * self.tamaño_pool):(j * self.tamaño_pool + self.tamaño_pool)]
                yield region, i, j

    def forward(self, entrada):
        self.ultima_entrada = entrada
        h, w, num_filtros = entrada.shape
        salida = np.zeros((h // self.tamaño_pool, w // self.tamaño_pool, num_filtros))

        for region, i, j in self._obtener_regiones(entrada):
            salida[i, j] = np.amax(region, axis=(0, 1))
        return salida

    def backward(self, d_L_d_salida):
        # --- EL RETO MATEMÁTICO ---
        # Solo el píxel que fue el máximo recibe el error
        d_L_d_entrada = np.zeros(self.ultima_entrada.shape)
        for region, i, j in self._obtener_regiones(self.ultima_entrada):
            h, w, f = region.shape
            amax = np.amax(region, axis=(0, 1))

            for i2 in range(h):
                for j2 in range(w):
                    for f2 in range(f):
                        if region[i2, j2, f2] == amax[f2]:
                            d_L_d_entrada[i*self.tamaño_pool+i2, j*self.tamaño_pool+j2, f2] = d_L_d_salida[i, j, f2]
        return d_L_d_entrada

# ==========================================
# 3. Capa Softmax (Para la salida densa final)
# ==========================================
class CapaSoftmax:
    def __init__(self, tamaño_entrada, tamaño_salida):
        self.pesos = np.random.randn(tamaño_entrada, tamaño_salida) / tamaño_entrada
        self.biases = np.zeros(tamaño_salida)

    def forward(self, entrada):
        self.ultima_entrada_shape = entrada.shape
        entrada_aplanada = entrada.flatten()
        self.ultima_entrada_aplanada = entrada_aplanada

        totales = np.dot(entrada_aplanada, self.pesos) + self.biases
        exp = np.exp(totales)
        self.ultima_salida = exp / np.sum(exp, axis=0)
        return self.ultima_salida

    def backward(self, d_L_d_salida, learning_rate):
        # --- EL RETO MATEMÁTICO ---
        # Calculamos gradientes para la capa densa final
        for i, gradiente in enumerate(d_L_d_salida):
            if gradiente == 0: continue

            t_exp = np.exp(self.ultima_salida)
            S_total = np.sum(t_exp)

            # Gradiente de softmax con respecto a los totales
            d_out_d_t = -t_exp[i] * t_exp / (S_total**2)
            d_out_d_t[i] = t_exp[i] * (S_total - t_exp[i]) / (S_total**2)

            # Gradiente de error con respecto a los totales
            d_L_d_t = gradiente * d_out_d_t

            # Gradientes de error con respecto a pesos/biases/entrada
            d_t_d_w = self.ultima_entrada_aplanada
            d_t_d_b = 1
            d_t_d_inputs = self.pesos

            # Actualizamos pesos y biases
            self.pesos -= learning_rate * np.dot(d_t_d_w[np.newaxis].T, d_L_d_t[np.newaxis])
            self.biases -= learning_rate * d_t_d_b * d_L_d_t

            # Retornamos el gradiente de error con respecto a la entrada (para la capa anterior)
            return np.dot(d_t_d_inputs, d_L_d_t).reshape(self.ultima_entrada_shape)

# ==========================================
# 4. Probando la CNN "Frankenstein"
# ==========================================
if __name__ == "__main__":
    # Creamos una imagen sintética de 28x28 (tipo MNIST)
    imagen = np.random.rand(28, 28)
    etiqueta = np.zeros(10)
    etiqueta[5] = 1 # Supongamos que es el número '5'

    # Instanciamos nuestras capas
    conv = CapaConvolucion(num_filtros=8, tamaño_kernel=3)
    pool = CapaMaxPooling(tamaño_pool=2)
    # 28-3+1 = 26. 26/2 = 13. 13*13*8 = 1352 neuronas de entrada densa
    softmax = CapaSoftmax(tamaño_entrada=13*13*8, tamaño_salida=10)

    # --- FORWARD PASS ---
    out_conv = conv.forward(imagen)
    out_pool = pool.forward(out_conv)
    prediccion = softmax.forward(out_pool)
    print("Predicción (probabilidades):", np.round(prediccion, 2))

    # --- CALCULO DE PÉRDIDA ---
    loss = -np.log(prediccion[np.argmax(etiqueta)])
    print("Pérdida (Cross-Entropy):", loss)

    # --- BACKWARD PASS ---
    # Gradiente inicial del error con respecto a la salida
    gradient = np.zeros(10)
    gradient[np.argmax(etiqueta)] = -1 / prediccion[np.argmax(etiqueta)]

    grad_softmax = softmax.backward(gradient, learning_rate=0.01)
    grad_pool = pool.backward(grad_softmax)
    conv.backward(grad_pool, learning_rate=0.01)
    print("Pesos actualizados. ¡La CNN aprendió un poquito!")


## Complejidad

Complejidad Temporal

* Entrenamiento: O(capas · filtros · tamaño_imagen · kernel²)
    * Depende del número de capas, filtros y tamaño de la imagen.

Complejidad Espacial

* Espacio: O(parámetros)
    * Se almacenan filtros, pesos y activaciones de la red.